In [1]:
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression, SGDRegressor, Ridge, LogisticRegression, Lasso
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, classification_report, roc_auc_score
import joblib
import pandas as pd
import numpy as np

In [2]:
# 线性回归直接预测房子价格
"""
    fetch_california_housing()：是函数，不是类；函数调用加括号执行逻辑，返回一个对象
        类比：load_iris()、fetch_20newsgroups()全部都是函数，不是类
        lb：这是函数运行完返回的一个Bunch对象，lb接收这个Bunch实例
        sklearn.utils.Bunch是sklearn自定义的容器类，像字典，但可以用 . 访问属性
"""
lb = fetch_california_housing(data_home='data')

print(f'获取特征值：{lb.data.shape}')
print('--' * 50)
print(lb.data[0])
print(f'目标值：{lb.target}')
print(lb.DESCR)
print('--' * 50)
print(lb.feature_names)

获取特征值：(20640, 8)
----------------------------------------------------------------------------------------------------
[   8.3252       41.            6.98412698    1.02380952  322.
    2.55555556   37.88       -122.23      ]
目标值：[4.526 3.585 3.521 ... 0.923 0.847 0.894]
.. _california_housing_dataset:

California Housing dataset
--------------------------

**Data Set Characteristics:**

:Number of Instances: 20640

:Number of Attributes: 8 numeric, predictive attributes and the target

:Attribute Information:
    - MedInc        median income in block group
    - HouseAge      median house age in block group
    - AveRooms      average number of rooms per household
    - AveBedrms     average number of bedrooms per household
    - Population    block group population
    - AveOccup      average number of household members
    - Latitude      block group latitude
    - Longitude     block group longitude

:Missing Attribute Values: None

This dataset was obtained from the StatLib reposi

In [3]:
# lb.data.shape中样本数也是20640
lb.target.shape

(20640,)

In [4]:
x_train, x_test, y_train, y_test = train_test_split(lb.data, lb.target, test_size=0.25, random_state=1)
print(x_train.shape)
"""
为什么要对特征X进行标准化（梯度类模型：线性回归SGD、神经网络）：
    不同特征单位、量级差别巨大
    举例：
        特征A：房屋面积0～200
        特征B：房间数量1～5
    梯度下降更新权重的时候：
        面积这个特征数值大 -> 损失对它的梯度天然更大，权重更新幅度剧烈
        房间数数值小 -> 梯度很小，更新很慢
    后果：
        损失曲面变成“扁椭圆”，梯度来回震荡，迭代很久才收敛，甚至不收敛
    标准化后所有特征均值为0，方差为1，各个特征梯度在同一个量级，下降稳定，收敛更快
"""
std_x = StandardScaler()
x_train = std_x.fit_transform(x_train)
x_test = std_x.transform(x_test)
"""
y_train.reshape(-1, 1):
    y_train 一般是一维数组：(n_samples, )，比如标签向量
    reshape(-1, 1)：把它变成二维：(n_samples, 1)
        -1：自动计算样本数量
        1:代表只有1列（一个特征）
"""
std_y = StandardScaler()
temp = y_train.reshape(-1, 1)

y_train = std_y.fit_transform(y_train.reshape(-1, 1))
print(y_train.shape)
y_test = std_y.transform(y_test.reshape(-1, 1))
print(y_test.shape)

(15480, 8)
(15480, 1)
(5160, 1)


In [7]:
test1 = np.array([1, 2, 3])
print(test1.shape)
print(type(test1))
test1.reshape(-1, 1)

(3,)
<class 'numpy.ndarray'>


array([[1],
       [2],
       [3]])

In [ ]:
lr = LinearRegression()

lr.fit(x_train, y_train)

# coefficient: n.（数）系数；（物理）率，系数
print('回归系数', lr.coef_)

y_predict = lr.predict(x_test)

y_lr_predict = std_y.inverse_transform(y_predict)

joblib.dump(lr, "./tmp/test.pkl")
print("正规方程测试集里面每个房子的预测价格：", y_predict[0: 10])

print("正规方程的均方误差：", mean_squared_error(y_test, y_predict))